In [1]:
import numpy as np
import pandas as pd


In [6]:
df_el= pd.read_csv("./extinct_languages.csv")

In [10]:
df_el.shape
df_el.info()
df_el.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           2722 non-null   int64  
 1   Name in English              2722 non-null   object 
 2   Name in French               2699 non-null   object 
 3   Name in Spanish              2701 non-null   object 
 4   Countries                    2721 non-null   object 
 5   Country codes alpha 3        2721 non-null   object 
 6   ISO639-3 codes               2458 non-null   object 
 7   Degree of endangerment       2722 non-null   object 
 8   Alternate names              1583 non-null   object 
 9   Name in the language         27 non-null     object 
 10  Number of speakers           2539 non-null   float64
 11  Sources                      2079 non-null   object 
 12  Latitude                     2719 non-null   float64
 13  Longitude         

,ID,Name in English,Name in French,Name in Spanish,Countries,Country codes alpha 3,ISO639-3 codes,Degree of endangerment,Alternate names,Name in the language,Number of speakers,Sources,Latitude,Longitude,Description of the location
0,1022,South Italian,italien du sud,napolitano-calabrés,Italy,ITA,nap,Vulnerable,Neapolitan; Neapolitan-Calabrese; неаполитанск...,NaN,7500000.0,NaN,40.9798,15.2490,"Campania, Lucania (Basilicata), Abruzzi (Abruz..."
1,1023,Sicilian,sicilien,siciliano,Italy,ITA,scn,Vulnerable,NaN,NaN,5000000.0,NaN,37.4399,14.5019,"Sicily (Sicilia), southern and central Calabri..."
2,383,Low Saxon,bas-saxon,bajo sajón,"Germany, Denmark, Netherlands, Poland, Russian...","DEU, DNK, NLD, POL, RUS","act, drt, frs, gos, nds, sdz, stl, twd, vel, wep",Vulnerable,"Low German, Niedersächsisch, Nedersaksisch, Ni...",Neddersassisch,4800000.0,NaN,53.4029,10.3601,"northern Germany, the north-eastern part of th..."
3,335,Belarusian,biélorusse,bielorruso,"Belarus, Latvia, Lithuania, Poland, Russian Fe...","BRB, LVA, LTU, POL, RUS, UKR",bel,Vulnerable,NaN,NaN,4000000.0,Hienadź Cychun: Weißrussisch. — Lexikon der Sp...,53.9560,27.5756,Belarus except the Polesian-speaking south-wes...
4,382,Lombard,lombard,lombardo,"Italy, Switzerland","ITA, CHE",lmo,Definitely endangered,NaN,NaN,3500000.0,NaN,45.7215,9.3273,the region of Lombardy (except the southernmos...


### 1. PREPARACIÓN DATASET COUNTRIES

In [2]:
df_cof= pd.read_csv("./countries of the world.csv")
df_cof.shape
df_cof.info()
df_cof_treated = df_cof.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227 entries, 0 to 226
Data columns (total 20 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Country                             227 non-null    object 
 1   Region                              227 non-null    object 
 2   Population                          227 non-null    int64  
 3   Area (sq. mi.)                      227 non-null    int64  
 4   Pop. Density (per sq. mi.)          227 non-null    object 
 5   Coastline (coast/area ratio)        227 non-null    object 
 6   Net migration                       224 non-null    object 
 7   Infant mortality (per 1000 births)  224 non-null    object 
 8   GDP ($ per capita)                  226 non-null    float64
 9   Literacy (%)                        209 non-null    object 
 10  Phones (per 1000)                   223 non-null    object 
 11  Arable (%)                          225 non-n

#### 1.1 Limpieza de formato de datos y columnas
- [ ] Eliminar espacios en blanco en nombres de columnas. Estandarizar nombres a formato snake_case.
- [ ] Comprobar si los separadores decimales están en formato correcto (algunas versiones usan coma en vez de punto).
- [ ] Conversión de tipos
- [ ] Convertir a `category`:
  - Country
  - Region
  - Climate (variable categórica codificada)

In [3]:
# columns in snake_case
df_cof_treated.rename(columns={'Area (sq. mi.)': 'Area_sq_mi',
    'Pop. Density (per sq. mi.)': 'Pop_density',
    'Coastline (coast/area ratio)': 'Coastline',
    'Net migration': 'Migration',
    'Infant mortality (per 1000 births)': 'Infant_mortality',
    'GDP ($ per capita)': 'GDP',
    'Literacy (%)': 'Literacy',
    'Phones (per 1000)': 'Phones',
    'Arable (%)': 'Arable',
    'Crops (%)': 'Crops',
    'Other (%)': 'Other'
    },
    inplace=True
)

In [4]:
def search_cols_with_commas(df):
    cols = []
    for col in df.columns:
        if df[col].dtype == 'object' and df[col].str.contains(',', regex=False, na=False).any():
            cols.append(col)
    return cols

cols_with_commas = search_cols_with_commas(df_cof_treated)
print(cols_with_commas)

['Country', 'Pop_density', 'Coastline', 'Migration', 'Infant_mortality', 'Literacy', 'Phones', 'Arable', 'Crops', 'Other', 'Climate', 'Birthrate', 'Deathrate', 'Agriculture', 'Industry', 'Service']


In [5]:
del cols_with_commas[0] # Eliminamos la columan Country porque realmente es string
print(cols_with_commas)

['Pop_density', 'Coastline', 'Migration', 'Infant_mortality', 'Literacy', 'Phones', 'Arable', 'Crops', 'Other', 'Climate', 'Birthrate', 'Deathrate', 'Agriculture', 'Industry', 'Service']


In [6]:
# Convertir los valores numéricos de string a float
for col in cols_with_commas:
    df_cof_treated[col] = (
        df_cof_treated[col]
        .str.replace(',', '.', regex=False)
        .astype(float)
    )

In [7]:
df_cof_treated[cols_with_commas].dtypes

Pop_density         float64
Coastline           float64
Migration           float64
Infant_mortality    float64
Literacy            float64
Phones              float64
Arable              float64
Crops               float64
Other               float64
Climate             float64
Birthrate           float64
Deathrate           float64
Agriculture         float64
Industry            float64
Service             float64
dtype: object

Para optimizar operaciones como groupby, value_counts. 
Evitar errores al no permitir texto libre. 
Especificar que no tienen carácter numérico
- [ ] Convertir a `category` :
  - Country
  - Region
  - Climate (variable categórica codificada)

In [8]:
#df_cof_treated["Country"] = df_cof_treated["Country"].astype("category")
#df_cof_treated["Region"] = df_cof_treated["Region"].astype("category")
# para categorizar la columna clima nos quedaremos con el primer valor ya que no es un factor con suficiente peso en nuestro EDA
df_cof_treated["Climate"] = (
    df_cof_treated["Climate"]
    .astype(str)
    .str.split(",", n=1)
    .str[0]
    .str.strip()
    .replace({"nan": None})
    .astype("category")
)
df_cof_treated.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227 entries, 0 to 226
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Country           227 non-null    object  
 1   Region            227 non-null    object  
 2   Population        227 non-null    int64   
 3   Area_sq_mi        227 non-null    int64   
 4   Pop_density       227 non-null    float64 
 5   Coastline         227 non-null    float64 
 6   Migration         224 non-null    float64 
 7   Infant_mortality  224 non-null    float64 
 8   GDP               226 non-null    float64 
 9   Literacy          209 non-null    float64 
 10  Phones            223 non-null    float64 
 11  Arable            225 non-null    float64 
 12  Crops             225 non-null    float64 
 13  Other             225 non-null    float64 
 14  Climate           205 non-null    category
 15  Birthrate         224 non-null    float64 
 16  Deathrate         223 non-

In [9]:
df_cof_treated.head()

,Country,Region,Population,Area_sq_mi,Pop_density,Coastline,Migration,Infant_mortality,GDP,Literacy,Phones,Arable,Crops,Other,Climate,Birthrate,Deathrate,Agriculture,Industry,Service
0,Afghanistan,ASIA (EX. NEAR EAST),31056997,647500,48.0,0.00,23.06,163.07,700.0,36.0,3.2,12.13,0.22,87.65,1.0,46.60,20.34,0.380,0.240,0.380
1,Albania,EASTERN EUROPE,3581655,28748,124.6,1.26,-4.93,21.52,4500.0,86.5,71.2,21.09,4.42,74.49,3.0,15.11,5.22,0.232,0.188,0.579
2,Algeria,NORTHERN AFRICA,32930091,2381740,13.8,0.04,-0.39,31.00,6000.0,70.0,78.1,3.22,0.25,96.53,1.0,17.14,4.61,0.101,0.600,0.298
3,American Samoa,OCEANIA,57794,199,290.4,58.29,-20.71,9.27,8000.0,97.0,259.5,10.00,15.00,75.00,2.0,22.46,3.27,NaN,NaN,NaN
4,Andorra,WESTERN EUROPE,71201,468,152.1,0.00,6.60,4.05,19000.0,100.0,497.2,2.22,0.00,97.78,3.0,8.71,6.25,NaN,NaN,NaN


#### 1.2 Limpieza de formato de datos y columnas
- [ ] Identificar columnas con `NaN`.
- [ ] Decidir estrategia:
  - Imputación (media/mediana) para variables numéricas.
  - Imputación por moda para categóricas.
  - Eliminación de filas si el porcentaje de nulos es bajo.
- [ ] Documentar la decisión (importante para reproducibilidad).

In [10]:
# Identificar columans con NaN
cols_con_nulos = df_cof_treated.columns[df_cof_treated.isna().any()]
cols_con_nulos

Index(['Migration', 'Infant_mortality', 'GDP', 'Literacy', 'Phones', 'Arable',
       'Crops', 'Other', 'Climate', 'Birthrate', 'Deathrate', 'Agriculture',
       'Industry', 'Service'],
      dtype='object')

In [11]:
def replace_nan_by_mean(df, columns, group_col="Region"):
    for col in columns:
        # Sólo reemplazar si la columna es numérica
        if df[col].dtype.kind in "fi":  # float o int
            df[col] = df[col].fillna(
                df.groupby(group_col)[col].transform("mean")
            )
    return df

In [12]:
df_cof_treated = replace_nan_by_mean(df_cof_treated, cols_con_nulos)

df_cof_treated.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227 entries, 0 to 226
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Country           227 non-null    object  
 1   Region            227 non-null    object  
 2   Population        227 non-null    int64   
 3   Area_sq_mi        227 non-null    int64   
 4   Pop_density       227 non-null    float64 
 5   Coastline         227 non-null    float64 
 6   Migration         227 non-null    float64 
 7   Infant_mortality  227 non-null    float64 
 8   GDP               227 non-null    float64 
 9   Literacy          227 non-null    float64 
 10  Phones            227 non-null    float64 
 11  Arable            227 non-null    float64 
 12  Crops             227 non-null    float64 
 13  Other             227 non-null    float64 
 14  Climate           205 non-null    category
 15  Birthrate         227 non-null    float64 
 16  Deathrate         227 non-



## 3. Conversión de tipos
- [ ] Convertir a `float`:
  - Area, Pop_density, Coastline, Net_migration
  - Infant_mortality, GDP, Literacy
  - Phones, Arable, Crops, Other
  - Agriculture, Industry, Service
  - Birthrate, Deathrate
- [ ] Convertir a `int` o `float` según convenga:
  - Population (tratada como continua)
- [ ] Convertir a `category`:
  - Country
  - Region
  - Climate (variable categórica codificada)

## 4. Manejo de valores faltantes
- [ ] Identificar columnas con `NaN`.
- [ ] Decidir estrategia:
  - Imputación (media/mediana) para variables numéricas.
  - Imputación por moda para categóricas.
  - Eliminación de filas si el porcentaje de nulos es bajo.
- [ ] Documentar la decisión (importante para reproducibilidad).

## 5. Detección de outliers
- [ ] Revisar outliers en:
  - GDP per capita
  - Infant mortality
  - Phones per 1000
  - Birthrate / Deathrate
- [ ] Usar boxplots o IQR para identificar valores extremos.
- [ ] Decidir si se corrigen, se winsorizan o se mantienen.

## 6. Coherencia interna
- [ ] Verificar que:
  - Arable + Crops + Other ≈ 100%
  - Agriculture + Industry + Service ≈ 100%
- [ ] Detectar países con sumas incoherentes y documentar.

## 7. Normalización y escalado (si se usará en modelos)
- [ ] Aplicar StandardScaler o MinMaxScaler a:
  - GDP, Population, Phones, Literacy, Birthrate, Deathrate
- [ ] No escalar variables categóricas ni proporciones que ya están en 0–100.

## 8. Creación de nuevas variables útiles (opcional)
- [ ] GDP_total = GDP_per_capita * Population
- [ ] Dependency_ratio (si se dispone de datos externos)
- [ ] Índice socioeconómico compuesto (si se usa en tu proyecto)

## 9. Validación final
- [ ] Revisar `df.describe()` para confirmar rangos coherentes.
- [ ] Comprobar que no quedan columnas en formato string cuando deberían ser numéricas.
- [ ] Guardar dataset limpio en CSV o Parquet para reproducibilidad.